# 🔮 EducaStock — Previsão de Consumo por Produto (Prophet)

> **ML Caso de Uso 1** — ONG Casa da Criança  
> Modelo: [Prophet (Meta)](https://facebook.github.io/prophet/) para séries temporais  
> Alternativa de fallback: Média Móvel Ponderada  

## Fluxo
1. Autentica com Firebase Admin SDK
2. Exporta **todas** as `stock_movements` do Firestore (entradas + saídas + descartes + ajustes)
3. Separa saídas (consumo) de entradas para análise completa
4. Agrega consumo diário por produto (apenas saídas)
5. Treina um modelo Prophet por produto
6. Gera previsões semanais e mensais
7. Calcula sugestão de reposição considerando estoque atual
8. Escreve resultado na coleção `consumption_forecasts`

## Pré-requisitos
- Arquivo `serviceAccountKey.json` do Firebase Admin (upload na célula abaixo)
- Acesso ao projeto Firebase do EducaStock


In [ ]:
# ─── Instalar dependências ───────────────────────────────────────────────────────
!pip install prophet firebase-admin pandas numpy matplotlib --quiet

In [ ]:
# ─── Upload do arquivo de credenciais Firebase ──────────────────────────────
from google.colab import files
import json

print("Faça upload do arquivo serviceAccountKey.json do Firebase Admin SDK")
print("Acesse: Firebase Console → Configurações do Projeto → Contas de Serviço → Gerar nova chave privada")
uploaded = files.upload()
SERVICE_ACCOUNT_KEY = list(uploaded.keys())[0]
print(f"\n✅ Arquivo carregado: {SERVICE_ACCOUNT_KEY}")

In [ ]:
# ─── Inicializar Firebase Admin ──────────────────────────────────────────────
import firebase_admin
from firebase_admin import credentials, firestore

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_KEY)
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("✅ Firebase conectado com sucesso")

In [ ]:
# ─── Exportar TODAS as movimentações do Firestore (entradas + saídas) ────────
import pandas as pd
from datetime import datetime, timezone


def parse_timestamp(ts) -> datetime:
    """Converte Timestamp Firestore, datetime ou string ISO para datetime com tz."""
    if ts is None:
        return datetime.now(timezone.utc)
    if isinstance(ts, datetime):
        return ts if ts.tzinfo else ts.replace(tzinfo=timezone.utc)
    if isinstance(ts, str):
        return datetime.fromisoformat(ts.replace("Z", "+00:00"))
    # Firestore DatetimeWithNanoseconds / Timestamp object
    if hasattr(ts, "seconds"):
        return datetime.fromtimestamp(ts.seconds, tz=timezone.utc)
    if hasattr(ts, "timestamp"):
        return datetime.fromtimestamp(ts.timestamp(), tz=timezone.utc)
    return datetime.now(timezone.utc)


def export_all_movements(limit: int = 10000) -> pd.DataFrame:
    """Exporta TODAS as movimentações (entrada, saida, descarte, ajuste) do Firestore."""
    print(f"Exportando até {limit} movimentações...")

    docs = (
        db.collection("stock_movements")
        .order_by("performedAt")
        .limit(limit)
        .stream()
    )

    records = []
    skipped = 0
    for doc in docs:
        d = doc.to_dict()
        try:
            dt = parse_timestamp(d.get("performedAt"))
            records.append({
                "ds": dt.date(),
                "product_id": d.get("productId", ""),
                "product_name": d.get("productName", "Desconhecido"),
                "quantity": abs(int(d.get("quantity", 0))),
                "type": d.get("type", "saida").lower().strip(),
                "notes": d.get("notes", ""),
                "performed_by": d.get("performedBy", ""),
            })
        except Exception as e:
            skipped += 1
            if skipped <= 5:
                print(f"  ⚠️  Registro ignorado: {e}")

    if skipped > 5:
        print(f"  ⚠️  ... e mais {skipped - 5} registros ignorados")

    df = pd.DataFrame(records) if records else pd.DataFrame(
        columns=["ds", "product_id", "product_name", "quantity", "type", "notes", "performed_by"]
    )
    print(f"✅ {len(df)} movimentações exportadas ({skipped} ignoradas)")
    if not df.empty:
        print(f"   Tipos: {df['type'].value_counts().to_dict()}")
        print(f"   Período: {df['ds'].min()} → {df['ds'].max()}")
    return df


df_raw = export_all_movements(limit=10000)

In [ ]:
# ─── Exportar estoque atual dos lotes ──────────────────────────────────────────────
def export_current_stock() -> dict:
    """Retorna {productId: total_quantity} com base nos lotes disponíveis."""
    print("Exportando estoque atual (lotes ativos)...")
    docs = (
        db.collection("batches")
        .where("status", "==", "available")
        .stream()
    )
    stock = {}
    for doc in docs:
        d = doc.to_dict()
        pid = d.get("productId", "")
        qty = int(d.get("quantity", 0))
        stock[pid] = stock.get(pid, 0) + qty

    print(f"✅ Estoque atual: {len(stock)} produtos | {sum(stock.values())} unidades totais")
    return stock


current_stock = export_current_stock()

In [ ]:
# ─── Pré-processamento — Separar entradas de saídas ───────────────────────────
import numpy as np

EXIT_TYPES  = {"saida", "descarte"}   # consumo real da ONG
ENTRY_TYPES = {"entrada"}              # reposição / doação
ADJUST_TYPES = {"ajuste"}             # correções manuais


def aggregate_daily(df: pd.DataFrame, types: set) -> pd.DataFrame:
    """Agrega quantidade diária por produto para os tipos informados."""
    sub = df[df["type"].isin(types)].copy()
    if sub.empty:
        return pd.DataFrame(columns=["product_id", "product_name", "ds", "y"])
    sub["ds"] = pd.to_datetime(sub["ds"])
    return (
        sub.groupby(["product_id", "product_name", "ds"])["quantity"]
        .sum()
        .reset_index()
        .rename(columns={"quantity": "y"})
    )


df_daily        = aggregate_daily(df_raw, EXIT_TYPES)   # consumo → treinamento Prophet
df_entries_daily = aggregate_daily(df_raw, ENTRY_TYPES) # entradas → análise de reposição
df_adjust_daily  = aggregate_daily(df_raw, ADJUST_TYPES)

print(f"Saídas  — {df_daily['product_id'].nunique()} produtos, {df_daily['y'].sum():.0f} un. totais")
print(f"Entradas — {df_entries_daily['product_id'].nunique()} produtos, {df_entries_daily['y'].sum():.0f} un. totais")
print(f"Ajustes  — {df_adjust_daily['product_id'].nunique()} produtos, {df_adjust_daily['y'].sum():.0f} un. totais")
df_daily.head(10)

In [ ]:
# ─── Painel completo de movimentações (entradas vs. saídas) ─────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle("EducaStock — Painel de Movimentações", fontsize=14, fontweight="bold")

COLOR_MAP = {
    "entrada": "#2E7D32",
    "saida":   "#C53030",
    "descarte":"#E65100",
    "ajuste":  "#5C6BC0",
}

# --- Gráfico 1: Volume total por tipo ---
if not df_raw.empty:
    type_totals = df_raw.groupby("type")["quantity"].sum().sort_values(ascending=True)
    colors = [COLOR_MAP.get(t, "#78909C") for t in type_totals.index]
    axes[0].barh(type_totals.index, type_totals.values, color=colors, edgecolor="white")
    axes[0].set_xlabel("Unidades")
    axes[0].set_title("Volume Total por Tipo")
    for i, v in enumerate(type_totals.values):
        axes[0].text(v * 0.02 + 0.5, i, f"{int(v):,}", va="center", fontsize=9)

# --- Gráfico 2: Evolução mensal entradas vs. saídas ---
df_raw["mes"] = pd.to_datetime(df_raw["ds"]).dt.to_period("M")
monthly = (
    df_raw[df_raw["type"].isin(["entrada", "saida"])]
    .groupby(["mes", "type"])["quantity"]
    .sum()
    .unstack(fill_value=0)
)
if not monthly.empty:
    x = range(len(monthly))
    has_entrada = "entrada" in monthly.columns
    has_saida   = "saida"   in monthly.columns
    if has_entrada:
        axes[1].bar([i - 0.2 for i in x], monthly["entrada"], width=0.4,
                     label="Entradas", color="#2E7D32", alpha=0.85)
    if has_saida:
        axes[1].bar([i + (0.2 if has_entrada else 0) for i in x],
                     monthly["saida"], width=0.4,
                     label="Saídas", color="#C53030", alpha=0.85)
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels([str(p) for p in monthly.index], rotation=45, ha="right")
    axes[1].set_ylabel("Unidades")
    axes[1].set_title("Evolução Mensal")
    axes[1].legend()

# --- Gráfico 3: Balanço (entrada - saída) por mês ---
if not monthly.empty and has_entrada and has_saida:
    balance = monthly.get("entrada", 0) - monthly.get("saida", 0)
    bal_colors = ["#2E7D32" if v >= 0 else "#C53030" for v in balance.values]
    axes[2].bar(range(len(balance)), balance.values, color=bal_colors, alpha=0.85, edgecolor="white")
    axes[2].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[2].set_xticks(range(len(balance)))
    axes[2].set_xticklabels([str(p) for p in balance.index], rotation=45, ha="right")
    axes[2].set_ylabel("Unidades")
    axes[2].set_title("Balanço Mensal (Entrada − Saída)")
    pos_patch = mpatches.Patch(color="#2E7D32", label="Ganho")
    neg_patch = mpatches.Patch(color="#C53030", label="Déficit")
    axes[2].legend(handles=[pos_patch, neg_patch])

plt.tight_layout()
plt.savefig("movements_summary.png", dpi=120, bbox_inches="tight")
plt.show()

# Tabela resumo
print("\n📊 RESUMO DE MOVIMENTAÇÕES")
print(f"{'Tipo':<12} {'Registros':>10} {'Total un.':>12} {'% do total':>12}")
print("-" * 48)
total_qty = df_raw["quantity"].sum()
for t, grp in df_raw.groupby("type"):
    pct = grp["quantity"].sum() / total_qty * 100 if total_qty else 0
    print(f"{t:<12} {len(grp):>10,} {grp['quantity'].sum():>12,.0f} {pct:>11.1f}%")
print(f"{'TOTAL':<12} {len(df_raw):>10,} {total_qty:>12,.0f}")

In [ ]:
# ─── Modelo Prophet por produto ───────────────────────────────────────────────
from prophet import Prophet
import warnings
warnings.filterwarnings("ignore")

MIN_DATA_POINTS = 10  # mínimo de dias com saída para treinar Prophet


def weighted_moving_average(series: pd.Series, window: int = 7) -> float:
    """Fallback: média móvel ponderada — pesos maiores para dados recentes."""
    if len(series) == 0:
        return 0.0
    n = min(len(series), window)
    recent = series.iloc[-n:].values
    weights = np.arange(1, n + 1, dtype=float)
    return float(np.dot(recent, weights) / weights.sum())


def compute_trend(forecast_df: pd.DataFrame) -> tuple:
    """Calcula tendência comparando segunda quinzena vs. primeira."""
    if len(forecast_df) < 14:
        return "stable", 0.0
    first  = forecast_df["yhat"].iloc[:15].mean()
    second = forecast_df["yhat"].iloc[15:].mean()
    if first == 0:
        return "stable", 0.0
    pct = ((second - first) / first) * 100
    if pct > 10:
        return "increasing", round(pct, 1)
    elif pct < -10:
        return "decreasing", round(abs(pct), 1)
    return "stable", round(abs(pct), 1)


def total_entries(product_id: str) -> float:
    """Soma total de entradas do produto no histórico."""
    rows = df_entries_daily[df_entries_daily["product_id"] == product_id]
    return float(rows["y"].sum()) if not rows.empty else 0.0


def train_and_forecast(
    product_id: str,
    product_name: str,
    df_product: pd.DataFrame,
) -> dict:
    """Treina Prophet (ou fallback) usando apenas saídas e retorna previsão."""
    n_points = len(df_product)
    stock    = current_stock.get(product_id, 0)
    entries  = total_entries(product_id)

    base = {
        "productId": product_id,
        "productName": product_name,
        "currentStock": stock,
        "totalHistoricalEntries": entries,
        "generatedAt": datetime.now(timezone.utc).isoformat(),
        "dataPoints": n_points,
    }

    if n_points < MIN_DATA_POINTS:
        daily_wma = weighted_moving_average(df_product["y"])
        weekly    = round(daily_wma * 7, 2)
        monthly   = round(daily_wma * 30, 2)
        suggested = max(0, int(monthly * 1.2) - stock)
        return {
            **base,
            "forecastWeekly": weekly,
            "forecastMonthly": monthly,
            "suggestedReplenishment": suggested,
            "ciLower": None,
            "ciUpper": None,
            "trend": "stable",
            "trendPercent": 0.0,
            "modelVersion": "moving_average_v1",
            "source": "moving_average",
        }

    model = Prophet(
        interval_width=0.80,
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=(n_points >= 365),
        changepoint_prior_scale=0.05,
    )
    model.fit(df_product[["ds", "y"]])

    future       = model.make_future_dataframe(periods=30)
    forecast     = model.predict(future)
    future_only  = forecast[forecast["ds"] > df_product["ds"].max()]
    weekly_rows  = future_only.head(7)
    monthly_rows = future_only.head(30)

    forecast_weekly  = max(0, weekly_rows["yhat"].sum())
    forecast_monthly = max(0, monthly_rows["yhat"].sum())
    ci_lower = max(0, weekly_rows["yhat_lower"].sum())
    ci_upper = max(0, weekly_rows["yhat_upper"].sum())

    trend, trend_pct = compute_trend(future_only)
    suggested = max(0, int(forecast_monthly * 1.2) - stock)

    return {
        **base,
        "forecastWeekly": round(forecast_weekly, 2),
        "forecastMonthly": round(forecast_monthly, 2),
        "suggestedReplenishment": suggested,
        "ciLower": round(ci_lower, 2),
        "ciUpper": round(ci_upper, 2),
        "trend": trend,
        "trendPercent": trend_pct,
        "modelVersion": "prophet_v1",
        "source": "prophet",
    }


print("✅ Funções de previsão definidas")

In [ ]:
# ─── Executar previsão para todos os produtos ─────────────────────────────────
results = []
products = df_daily.groupby(["product_id", "product_name"])

print(f"Treinando modelos para {len(products)} produtos com histórico de saída...\n")

for (pid, pname), group in products:
    group = group.sort_values("ds")
    try:
        result = train_and_forecast(pid, pname, group)
        results.append(result)
        tag = "Prophet" if result["source"] == "prophet" else "Média Móvel"
        print(f"  [{tag:<12}] {pname[:35]:<35}: "
              f"{result['forecastWeekly']:6.1f} un/sem | "
              f"estoque {result['currentStock']:>4} | "
              f"repor {result['suggestedReplenishment']:>4} un.")
    except Exception as e:
        print(f"  ❌ Erro em {pname}: {e}")

# Produtos com entradas mas sem saídas (apenas recebidos, nunca distribuídos)
only_entries = set(df_entries_daily["product_id"]) - set(df_daily["product_id"])
if only_entries:
    print(f"\n⚠️  {len(only_entries)} produto(s) com entrada mas sem saída registrada (não incluídos na previsão):")
    for pid in list(only_entries)[:5]:
        pname = df_entries_daily[df_entries_daily["product_id"] == pid]["product_name"].iloc[0]
        print(f"   - {pname}")

print(f"\n✅ {len(results)} produtos processados")

In [ ]:
# ─── Visualização das principais previsões ────────────────────────────────────
df_results = pd.DataFrame(results)

top_replenish = (
    df_results[df_results["suggestedReplenishment"] > 0]
    .sort_values("suggestedReplenishment", ascending=False)
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("EducaStock — Previsão de Consumo (Prophet)", fontsize=14, fontweight="bold")

# Gráfico 1: Sugestão de reposição
if not top_replenish.empty:
    colors = [
        "#C53030" if row["currentStock"] < row["forecastWeekly"]
        else "#B7791F"
        for _, row in top_replenish.iterrows()
    ]
    bars = axes[0].barh(
        top_replenish["productName"].str[:28],
        top_replenish["suggestedReplenishment"],
        color=colors, edgecolor="white"
    )
    axes[0].set_xlabel("Unidades a Repor")
    axes[0].set_title("Top Sugestões de Reposição")
    axes[0].invert_yaxis()
    urgente = mpatches.Patch(color="#C53030", label="Urgente (estoque < consumo/sem)")
    normal  = mpatches.Patch(color="#B7791F", label="Normal")
    axes[0].legend(handles=[urgente, normal], fontsize=8)

# Gráfico 2: Forecast mensal vs. estoque atual vs. entradas históricas
sample = df_results.nlargest(10, "forecastMonthly")
x = range(len(sample))
axes[1].bar([i - 0.25 for i in x], sample["forecastMonthly"], width=0.25,
             label="Previsão 30d (saída)", color="#C53030", alpha=0.85)
axes[1].bar([i        for i in x], sample["currentStock"], width=0.25,
             label="Estoque atual", color="#2E7D32", alpha=0.85)
axes[1].bar([i + 0.25 for i in x], sample["totalHistoricalEntries"], width=0.25,
             label="Entradas históricas", color="#1565C0", alpha=0.65)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels([n[:14] for n in sample["productName"]], rotation=45, ha="right")
axes[1].set_ylabel("Unidades")
axes[1].set_title("Previsão 30d vs Estoque vs Entradas")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("forecast_summary.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Gráfico salvo: forecast_summary.png")

In [ ]:
# ─── Gravar previsões no Firestore ──────────────────────────────────────────────
def write_forecasts_to_firestore(forecasts: list) -> None:
    col   = db.collection("consumption_forecasts")
    batch = db.batch()
    count = 0

    for f in forecasts:
        doc_ref = col.document(f["productId"])
        batch.set(doc_ref, f, merge=False)
        count += 1
        if count % 400 == 0:
            batch.commit()
            batch = db.batch()
            print(f"  Batch enviado: {count} registros...")

    batch.commit()
    print(f"✅ {count} previsões gravadas em consumption_forecasts")


write_forecasts_to_firestore(results)

In [ ]:
# ─── Resumo final ───────────────────────────────────────────────────────────
df_res = pd.DataFrame(results)

sep = "=" * 60
print(sep)
print(" RESUMO — PREVISÃO DE CONSUMO EDUCASTOCK")
print(sep)
print(f"Total de movimentações exportadas : {len(df_raw):>6,}")
print(f"  Entradas                         : {len(df_raw[df_raw['type']=='entrada']):>6,}")
print(f"  Saídas                           : {len(df_raw[df_raw['type']=='saida']):>6,}")
print(f"  Descartes                        : {len(df_raw[df_raw['type']=='descarte']):>6,}")
print(f"  Ajustes                          : {len(df_raw[df_raw['type']=='ajuste']):>6,}")
print(sep)
print(f"Produtos processados (com saída)  : {len(df_res):>6}")
print(f"  Com modelo Prophet               : {(df_res['source']=='prophet').sum():>6}")
print(f"  Com Média Móvel (fallback)       : {(df_res['source']=='moving_average').sum():>6}")
print(f"Produtos precisam reposição       : {(df_res['suggestedReplenishment']>0).sum():>6}")
print(f"Consumo total previsto / mês      : {df_res['forecastMonthly'].sum():>8.0f} un.")
print(f"Reposição total sugerida          : {df_res['suggestedReplenishment'].sum():>8.0f} un.")
print(sep)
print("\nTop 5 produtos para reposição urgente:")
urgent = df_res.nlargest(5, "suggestedReplenishment")[
    ["productName", "suggestedReplenishment", "currentStock",
     "forecastMonthly", "totalHistoricalEntries", "source"]
]
urgent.columns = ["Produto", "Repor", "Estoque", "Previsão30d", "EntradaHist", "Modelo"]
print(urgent.to_string(index=False))
print(f"\n✅ Previsões disponíveis no app EducaStock!")

## 📌 Instruções de uso

### Agendamento recomendado
Execute este notebook **1x por semana** ou após grandes movimentações de estoque.

### Índice Firestore necessário
Crie um índice composto no Firestore para a query de exportação:
```
Coleção: stock_movements
Campos:  performedAt (ASC)
```

### Coleção gerada: `consumption_forecasts`
```json
{
  "productId": "abc123",
  "productName": "Arroz",
  "forecastWeekly": 5.2,
  "forecastMonthly": 21.0,
  "currentStock": 10,
  "suggestedReplenishment": 15,
  "totalHistoricalEntries": 120.0,
  "ciLower": 3.5,
  "ciUpper": 7.0,
  "trend": "increasing",
  "trendPercent": 12.3,
  "modelVersion": "prophet_v1",
  "generatedAt": "2025-06-01T00:00:00Z",
  "dataPoints": 45,
  "source": "prophet"
}
```

### Tipos de movimentação suportados
| tipo | uso no modelo |
|------|---------------|
| `entrada` | Exibido como dado histórico de reposição |
| `saida` | Usado para treinar a previsão de consumo |
| `descarte` | Computado junto com saída |
| `ajuste` | Exportado mas não entra no modelo |

### Boas práticas
- Nunca usar dados pessoais (usuário, endereço) no treinamento
- Manter baseline simples (média histórica) para comparar com Prophet
- Resultado é sempre **sugestão** — decisão final é do usuário
- Registrar previsões e feedback no Firestore para melhorar o modelo
